# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [ ]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [ ]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "adk1-496817"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'adk1-496817' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '824f091d-afa9-4463-bc80-c568e0f9531b'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable and enjoyable! This plan focuses on the vibrant arts scene and serene green spaces found in nearby San Jose.

***

### **Relaxing & Artsy Day Trip to San Jose (Affordable)**

**Mood:** Relaxing, Artsy
**Budget:** Affordable (focus on free activities, low-cost options, and recommendations for budget-friendly dining)

---

#### **Morning (10:00 AM - 12:30 PM): Serenity in the Japanese Friendship Garden**

Start your day with tranquility at the **Japanese Friendship Garden** in San Jose. This beautiful six-acre garden is patterned after Japan's famous Korakuen Garden in Okayama, featuring three main ponds stocked with ko

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable and enjoyable! This plan focuses on the vibrant arts scene and serene green spaces found in nearby San Jose.

***

### **Relaxing & Artsy Day Trip to San Jose (Affordable)**

**Mood:** Relaxing, Artsy
**Budget:** Affordable (focus on free activities, low-cost options, and recommendations for budget-friendly dining)

---

#### **Morning (10:00 AM - 12:30 PM): Serenity in the Japanese Friendship Garden**

Start your day with tranquility at the **Japanese Friendship Garden** in San Jose. This beautiful six-acre garden is patterned after Japan's famous Korakuen Garden in Okayama, featuring three main ponds stocked with koi fish, picturesque bridges, and waterfalls. It offers a peaceful retreat and an opportunity to appreciate landscape artistry.

*   **Activity:** Stroll through the meticulously designed pathways, admire the koi ponds, and enjoy the serene atmosphere.
*   **Cost:** Free admission to the garden. Note that there is a parking fee for Kelley Park, where the garden is located.
*   **Operating Hours:** The Japanese Friendship Garden is open daily from 10:00 AM to 7:00 PM during the summer season (Memorial Day to Labor Day).
*   **Location:** 1300 Senter Rd, San Jose, CA 95112.

#### **Lunch (12:30 PM - 1:30 PM): Affordable Eats in Downtown San Jose**

Head towards downtown San Jose, specifically the SoFA (South First Area) district, which is known for its artistic vibe and diverse eateries. You'll find a variety of casual and affordable restaurants offering everything from ethnic cuisine to classic American fare. Look for local delis, taquerias, or cafes that fit your taste and budget.

*   **Recommendation:** Explore the South First Street area for numerous budget-friendly lunch options.

#### **Afternoon (1:30 PM - 5:00 PM): Immerse in Contemporary Art & Public Murals**

After lunch, delve into San Jose's contemporary art scene, which is concentrated in the SoFA district.

*   **Institute of Contemporary Art San José (ICA San José):** Begin with a visit to the ICA San José, a hub for innovative visual art exhibitions.
    *   **Activity:** Explore the rotating exhibitions featuring contemporary artists. The ICA is known for engaging Bay Area audiences with socially relevant art.
    *   **Cost:** Admission is always free.
    *   **Operating Hours:** Open Thursday to Sunday, 12:00 PM to 5:00 PM.
    *   **Location:** 560 South First Street, San José, CA 95113.

*   **KALEID Gallery (Optional Stop):** Just a short walk away, KALEID Gallery showcases fine art, limited editions, and creative gifts from local artists. Browsing is free, and you might find unique, affordable pieces if you're looking to take a souvenir home.
    *   **Location:** 320 S 1st Street, San Jose, CA 95113.

*   **Self-Guided Downtown Public Art Walk:** The SoFA district and surrounding downtown area are rich with public art, including impressive murals and sculptures.
    *   **Activity:** Use a self-guided map (available online from San Jose tourism or public art websites) to explore the vibrant street art. Look for pieces like "The Arch of Dignity, Equality and Justice" or the "Olympic Black Power Statue."
    *   **Cost:** Free.

#### **Evening (5:00 PM - 7:00 PM): Relaxed Dinner & Evening Stroll**

Conclude your artsy and relaxing day with an enjoyable evening in downtown San Jose.

*   **Dinner:** Choose from the diverse dining options still available in the SoFA district or explore other parts of downtown San Jose for a relaxed dinner that fits your budget. Many cafes and casual restaurants offer outdoor seating, perfect for a pleasant evening.
*   **Evening Stroll:** Take a leisurely walk to revisit some of the public art installations, many of which are beautifully lit at night, or simply enjoy the evening ambiance of the SoFA district.

---

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '81d5aac2-eb8a-4552-8090-391d06fd8f00'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="Lake Tahoe is a large area, and the weather can vary depending on the specific location. Could you please tell me which city near Lake Tahoe you're interested in for the weather forecast?",
      thought_signature=b'\n\x84\x03\x01\x8f=k_\xb3{\xddp\x1f\x99\xc0\xfe\xa2\xef\x82\xdb<\x1f\x8b\xbe\xab\x08\x86)\xb4@\xc25\xae\x94WLT\xaaO\x99Y\xa8\xcd\xe2\x19\xa4e4\xe3\xefG}\x14\xe6\x0b\x12?\xe0Y\xc6\xf7\xc3E\xee?-}H\x1a\x94\xe9\xe6^zm\xedn\x8f\xffTe\xce\xf8y \xf7\x1d\x8d\xb8\xfb\x97.\xf5D\xe4\xd6\xd2...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateC

Lake Tahoe is a large area, and the weather can vary depending on the specific location. Could you please tell me which city near Lake Tahoe you're interested in for the weather forecast?

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [ ]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [ ]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '60e420e1-4291-4271-93aa-a4e91f54b350'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-f03f5fde-ce3b-4999-b688-1728691ac2e5',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xdc\x03\x01\x8f=k_\x1c\x10o\x8e\xbfM\x1d\xa8\x04ifR\xc1+\x90\x7f\x07\xe0\xed\xe9\x86\xd9\x8f\xa3\x154Q2\nb\xd9-\x01\x02t\xcf\x822\x97Y\x9f\xd7\x95\xc2\x8ai+1*\xc8e\x9c\x8c\xa1\x08\xf2\tdj5*LtSI_;\xb3\x18\xc9\xf1\xb1\x18\x0b\xd4&\xcfu\xac\xbau\x7fzr|b\xd6U\x1b...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted

The top-rated hotels in San Francisco are The Grand Hotel (5-star, 450 reviews) and Seaside Inn (4-star, 620 reviews).

The hotel with the most reviews is the Seaside Inn. For a dinner spot near the Seaside Inn in San Francisco, I recommend Gary Danko.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [ ]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [ ]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: eea63364-904c-4375-b1c6-0ef8d58a46f3

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'eea63364-904c-4375-b1c6-0ef8d58a46f3'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! Lisbon is a fantastic choice with its rich history and delicious cuisine. I can definitely help you plan a wonderful 2-day trip.

Let's start by planning **Day 1**. How does this sound for your first day, keeping in mind your interest in historic sites and local food?

### Day 1: Historic Alfama & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the narrow, winding streets of **Alfama**, Lisbon's oldest district.
    *   Visit **Lisbon Cathedral (Sé de Lisboa)**, 

Hello! Lisbon is a fantastic choice with its rich history and delicious cuisine. I can definitely help you plan a wonderful 2-day trip.

Let's start by planning **Day 1**. How does this sound for your first day, keeping in mind your interest in historic sites and local food?

### Day 1: Historic Alfama & Culinary Delights

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District**
    *   Begin your day by getting lost in the narrow, winding streets of **Alfama**, Lisbon's oldest district.
    *   Visit **Lisbon Cathedral (Sé de Lisboa)**, a magnificent Romanesque cathedral with a history dating back to the 12th century.
    *   Climb up to **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for breathtaking panoramic views of the Tagus River and the Alfama rooftops.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine**
    *   Enjoy a traditional Portuguese lunch in Alfama. Look for a local "tasca" (a traditional, informal restaurant) to try dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or *Sardinhas Assadas* (grilled sardines, especially if you're there in summer).
*   **Afternoon (2:30 PM - 6:00 PM): São Jorge Castle**
    *   Head to **São Jorge Castle (Castelo de São Jorge)**, an iconic historic castle offering stunning views, peacocks roaming the grounds, and a fascinating insight into Lisbon's past. Explore its ramparts, gardens, and the archaeological site.
*   **Evening (7:00 PM onwards): Dinner & Fado Experience**
    *   Savor dinner in the Alfama district, perhaps trying another local specialty.
    *   Consider experiencing a traditional **Fado show**, a UNESCO-recognized musical genre unique to Portugal, often found in the Alfama and Bairro Alto districts.

How does this sound for your first day in Lisbon? We can adjust anything you like!

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'eea63364-904c-4375-b1c6-0ef8d58a46f3'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""You're right to point that out! My apologies for overlooking your preference regarding castles in the afternoon while focusing on the morning. Let's completely revise Day 1 to ensure it aligns perfectly with your interests in historical sites (without castles) and great local food.

Here's a new proposal for Day 1:

### Day 1 (Revised): Belém's Maritime History & City Center Exploration

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime Heritage**
    *   Begin your day in the historic district of **Belém**, a crucial point during Portugal's Age of Discovery.
    

You're right to point that out! My apologies for overlooking your preference regarding castles in the afternoon while focusing on the morning. Let's completely revise Day 1 to ensure it aligns perfectly with your interests in historical sites (without castles) and great local food.

Here's a new proposal for Day 1:

### Day 1 (Revised): Belém's Maritime History & City Center Exploration

*   **Morning (9:00 AM - 1:00 PM): Discover Belém's Maritime Heritage**
    *   Begin your day in the historic district of **Belém**, a crucial point during Portugal's Age of Discovery.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site known for its stunning Manueline architecture and intricate cloisters.
    *   See the iconic **Belém Tower (Torre de Belém)**, a fortified tower that served as a ceremonial gateway to Lisbon, and the **Monument to the Discoveries (Padrão dos Descobrimentos)**, which celebrates Portuguese explorers.
*   **Lunch (1:00 PM - 2:30 PM): Iconic Pastries & Local Bites in Belém**
    *   No trip to Belém is complete without trying the world-famous *Pastéis de Belém* (custard tarts) at the original bakery, which has been making them since 1837.
    *   Enjoy a light lunch or a traditional Portuguese sandwich at a local eatery in the Belém area.
*   **Afternoon (2:30 PM - 6:00 PM): Baixa, Chiado & Carmo Convent Ruins**
    *   Travel back towards the city center to explore the elegant **Baixa** district, renowned for its grid-like street plan and majestic squares, rebuilt after the 1755 earthquake.
    *   Ascend to the **Carmo Convent Ruins (Convento do Carmo)**, a striking Gothic church that was largely destroyed in the 1755 earthquake, now housing an archaeological museum. This offers a unique and poignant historical experience.
    *   Stroll through the vibrant **Chiado** neighborhood, known for its historic cafes, bookstores, and theaters.
*   **Evening (7:00 PM onwards): Dinner & Traditional Portuguese Flavors**
    *   Have dinner in the Baixa or Chiado area. Consider trying regional specialties like *Arroz de Marisco* (seafood rice) or *Polvo à Lagareiro* (roasted octopus with olive oil and potatoes), paired with a local Portuguese wine.
    *   Enjoy a relaxed evening, perhaps with a walk through the illuminated streets of central Lisbon.

How does this revised Day 1 plan sound to you? We've replaced the castle and the original morning activity with different, but equally rich, historical experiences.

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'eea63364-904c-4375-b1c6-0ef8d58a46f3'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's move on to planning your second and final day, keeping your interests in historic sites and great local food at the forefront, and ensuring we explore new parts of the city.

Here is a proposed itinerary for **Day 2**:

### Day 2: Alfama's Charm, Culinary Market & Panoramic Views

*   **Morning (9:00 AM - 1:00 PM): Discover the Authentic Alfama & Iconic Tram 28**
    *   Start your day by immersing yourself in **Alfama**, Lisbon's oldest and most atmospheric district. Wander its labyrinthine streets, soak in the medieval atmosphere, and discover hidden alleys. This area,

Excellent! I'm glad Day 1 is perfect. Let's move on to planning your second and final day, keeping your interests in historic sites and great local food at the forefront, and ensuring we explore new parts of the city.

Here is a proposed itinerary for **Day 2**:

### Day 2: Alfama's Charm, Culinary Market & Panoramic Views

*   **Morning (9:00 AM - 1:00 PM): Discover the Authentic Alfama & Iconic Tram 28**
    *   Start your day by immersing yourself in **Alfama**, Lisbon's oldest and most atmospheric district. Wander its labyrinthine streets, soak in the medieval atmosphere, and discover hidden alleys. This area, largely untouched by the 1755 earthquake, offers a glimpse into historic Lisbon.
    *   Visit the impressive **Lisbon Cathedral (Sé de Lisboa)**, a significant Romanesque monument with centuries of history, showcasing various architectural styles from different eras.
    *   Experience a quintessential Lisbon activity: take a ride on the iconic **Tram 28** (or at least part of its route). This vintage tram offers a scenic journey through some of Lisbon's most picturesque neighborhoods, including Alfama, Graça, and Estrela, passing by many historic sites and providing unique perspectives of the city.
    *   Ascend to **Miradouro de Santa Luzia** and **Miradouro das Portas do Sol** for postcard-perfect views over the Tagus River and the Alfama rooftops, ideal for capturing stunning photographs.
*   **Lunch (1:00 PM - 2:30 PM): Culinary Journey at Time Out Market**
    *   Head towards **Cais do Sodré** for a unique culinary experience at the **Time Out Market (Mercado da Ribeira)**. This vibrant food hall brings together some of Lisbon's best restaurants, chefs, and traditional market stalls under one roof. It's a fantastic opportunity to sample a wide variety of high-quality Portuguese dishes, from seafood to traditional pastries, all in one lively location.
*   **Afternoon (2:30 PM - 6:00 PM): Bairro Alto & Principe Real's Eclectic Charm**
    *   Ascend to the lively **Bairro Alto** district. By day, it's quieter than its nightlife persona suggests, allowing you to appreciate its charming streets, colorful buildings, and traditional architecture.
    *   Stroll towards **Principe Real**, a more upscale and elegant neighborhood known for its unique boutique shops, beautiful gardens (such as Jardim do Príncipe Real), and interesting 19th-century architecture.
    *   Visit the **Miradouro de São Pedro de Alcântara** for another spectacular panoramic view, this time overlooking the Baixa district and São Jorge Castle, providing a different perspective of the city.
*   **Evening (7:00 PM onwards): Farewell Dinner & Sunset Views**
    *   Enjoy a farewell dinner in **Bairro Alto** or **Principe Real**. These areas offer a wide range of dining options, from traditional Portuguese taverns to contemporary restaurants, many serving excellent food with a focus on local ingredients.
    *   Consider finding a restaurant with a view to savor your last evening in Lisbon, or simply enjoy a relaxed dinner amidst the charming evening atmosphere of these neighborhoods.

How does this plan for Day 2 sound to you?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [ ]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 692ac087-b32f-4612-b438-584ab7506631
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '692ac087-b32f-4612-b438-584ab7506631'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a wonderful 2-day trip focusing on historic sites and local cuisine.

Let's start with **Day 1**. How about this itinerary?

**Day 1: Historic Alfama & Baixa Districts**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama.** Start your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Visit

Great! Lisbon is a fantastic choice with its rich history and delicious food. I'll help you plan a wonderful 2-day trip focusing on historic sites and local cuisine.

Let's start with **Day 1**. How about this itinerary?

**Day 1: Historic Alfama & Baixa Districts**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama.** Start your day by wandering through the narrow, winding streets of Alfama, Lisbon's oldest district. Visit the **Lisbon Cathedral (Sé de Lisboa)** and then make your way up to **São Jorge Castle (Castelo de São Jorge)** for breathtaking panoramic views of the city.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine in Alfama.** Find a local tasca (a traditional, informal restaurant) in Alfama for some authentic Portuguese dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or *Sardinhas Assadas* (grilled sardines, especially in season).
*   **Afternoon (2:30 PM - 6:00 PM): Baixa District & Praça do Comércio.** Descend from Alfama into the Baixa district, rebuilt after the 1755 earthquake. Walk along Rua Augusta, admire the neoclassical architecture, and pass under the **Arco da Rua Augusta** to reach **Praça do Comércio**. Enjoy the riverside views and the grandeur of the square.
*   **Evening (7:30 PM onwards): Dinner & Fado in Alfama.** Return to Alfama for a traditional Portuguese dinner, perhaps followed by a live **Fado show**. Many restaurants in Alfama offer dinner and fado performances, providing a truly local cultural experience.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 28410e76-da1f-423f-9279-2ddf64ba2626
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '28410e76-da1f-423f-9279-2ddf64ba2626'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a plan for Day 2 in Tokyo, focusing on a mix of traditional culture, history, and a unique dining experience:

### Day 2: Traditional Tokyo & Artistic Exploration

*   **Morning (9:00 AM - 12:00 PM): Asakusa - Senso-ji Temple & Nakamise-dori**
    Start your day by immersing yourself in old Tokyo at Asakusa. Visit Senso-ji Temple, Tokyo's oldest temple, and stroll through Nakamise-dori, a vibrant market street leading to the temple, offering traditional snacks and souvenirs.
*   **Lunch (12:00 PM - 1:30 PM): Traditional Japanese Lunch in Asakusa**
    Enjoy a traditional Japanese lunch in the Asa

Here's a plan for Day 2 in Tokyo, focusing on a mix of traditional culture, history, and a unique dining experience:

### Day 2: Traditional Tokyo & Artistic Exploration

*   **Morning (9:00 AM - 12:00 PM): Asakusa - Senso-ji Temple & Nakamise-dori**
    Start your day by immersing yourself in old Tokyo at Asakusa. Visit Senso-ji Temple, Tokyo's oldest temple, and stroll through Nakamise-dori, a vibrant market street leading to the temple, offering traditional snacks and souvenirs.
*   **Lunch (12:00 PM - 1:30 PM): Traditional Japanese Lunch in Asakusa**
    Enjoy a traditional Japanese lunch in the Asakusa area. You can find many local eateries serving tempura, soba, or unagi (eel).
*   **Afternoon (1:30 PM - 5:30 PM): Ueno Park - Museums or Zoo**
    Head to Ueno Park, a vast public park home to several museums and a zoo. You can choose to explore the Tokyo National Museum, which houses an extensive collection of Japanese art and artifacts, or visit Ueno Zoo, famous for its giant pandas.
*   **Evening (6:00 PM onwards): Dinner & Drinks in Shinjuku Golden Gai or Omoide Yokocho**
    Experience a different side of Tokyo's nightlife in Shinjuku. For dinner, explore the narrow alleys of Omoide Yokocho (Memory Lane) for a casual and atmospheric meal with yakitori and other local delights. Afterward, you could venture into Golden Gai for a unique bar-hopping experience in its tiny, themed bars.

How does this sound for your second day?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
